# 🔴 Solution: Implement RMSNorm（面试版）

## 📋 题目描述

**难度：** Medium

实现 RMSNorm（均方根层归一化）。

RMSNorm 是 LayerNorm 的简化替代，跳过均值减法，仅通过激活值的均方根进行归一化。

**签名:** `rms_norm(x, weight, eps=1e-6) -> Tensor`

**参数:**
- `x` — 输入张量 (..., D)
- `weight` — 可学习的缩放参数 (D,)
- `eps` — 数值稳定性的 epsilon

**返回:** 归一化后的张量，形状与 x 相同

**约束:**
- `RMS(x) = sqrt(mean(x^2) + eps)` 沿最后一维
- 输出：`x / RMS(x) * weight`

**提示：** `rms = sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)`
`return x / rms * weight`  ← 无需减均值


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def rmsnorm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # ---- Step 1: 计算均方值 ----
    # RMS² = mean(x²)，沿最后一维
    # 与 LayerNorm 的区别：不减均值，直接算平方的均值
    # 数学上：RMS(x) = √(1/d × Σx_i²)
    rms_sq = x.pow(2).mean(dim=-1, keepdim=True)

    # ---- Step 2: 归一化 ----
    # x / RMS(x)：缩放到 RMS ≈ 1
    # eps=1e-6（比 LayerNorm 的 1e-5 更小，因为 RMS 通常更大）
    x_norm = x / torch.sqrt(rms_sq + eps)

    # ---- Step 3: 缩放 ----
    # 只有 γ 没有 β，这是 RMSNorm 的特点
    return weight * x_norm

# 面试追问：
# Q: RMSNorm vs LayerNorm？
# A: RMSNorm 不减均值，计算更快；只有 γ 没有 β
# Q: 为什么现代 LLM 用 RMSNorm？
# A: 计算效率更高，实验效果与 LayerNorm 相当

In [ ]:
x = torch.randn(2, 8)
out = rms_norm(x, torch.ones(8))
print('RMS of output:', out.pow(2).mean(dim=-1).sqrt())

## 📝 核心思路总结

1. **RMS 公式**：√(mean(x²))，不减均值，比 LayerNorm 更简洁
2. **只有 γ 没有 β**：减少参数，计算更快
3. **数值稳定**：eps 防止除零
4. **现代 LLM 标配**：LLaMA、GPT-NeoX 等都用 RMSNorm